# Лабораторная работа 4. Бонус — Проверка данных на нормальность

**ФИО:** Иноземцев Сергей Викторович  
**Группа:** ЕТ-216

Цель: реализовать функцию Gauss с критерием χ² Пирсона и исследовать распределения Iris.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 25)
pd.set_option('display.max_rows', 25)
sns.set_theme(style='whitegrid', font='DejaVu Sans')
plt.rcParams.update({'figure.figsize': (9, 5), 'figure.dpi': 110})
DATA = Path('.')
FIG = Path('figures')
FIG.mkdir(exist_ok=True)

def show(name):
    plt.tight_layout()
    plt.savefig(FIG / (name + '.png'), bbox_inches='tight')
    plt.show()
    plt.close('all')


## Бонус Iris Функция Gauss
H0: признак имеет нормальное распределение с оценёнными μ и σ; H1: распределение отличается. Критерий χ² Пирсона строится по равновероятным интервалам нормального распределения с крайними границами ±∞. Ожидаемая частота не меньше 5. Два параметра оценены по выборке, поэтому ddof=2 и число степеней свободы k−3. Применимость асимптотического приближения ограничена, особенно при малых выборках и округлении измерений.

In [ ]:
iris_raw=pd.read_excel(DATA/'Iris.xlsx',sheet_name='Data')
iris_raw['Species_name']=iris_raw['Species_name'].str.strip()
translations={'Petal_width':'Ширина лепестка','Petal_length':'Длина лепестка','Sepal_width':'Ширина чашелистика','Sepal_length':'Длина чашелистика'}
iris=iris_raw.drop(columns=['Species_No','Species_name']).rename(columns=translations);iris.index=np.arange(1,151);display(iris.head())
def Gauss(x,name):
    x=np.asarray(pd.Series(x).dropna(),dtype=float);n=len(x);k=min(8,n//10)
    mu=x.mean();sigma=x.std(ddof=0)
    if k<4 or sigma<=0:raise ValueError('Недостаточно данных для χ² с двумя оценёнными параметрами')
    edges=stats.norm.ppf(np.linspace(0,1,k+1),loc=mu,scale=sigma)
    observed,_=np.histogram(x,bins=edges);expected=np.full(k,n/k)
    chi,p=stats.chisquare(observed,expected,ddof=2)
    print(name,'n=',n,'mean=',mu,'median=',np.median(x),'skew=',stats.skew(x,bias=False),'excess=',stats.kurtosis(x,bias=False))
    print('H0: нормальность; H1: не нормальность; χ²=',chi,'df=',k-3,'p=',p,'; H0', 'отклонена' if p<.05 else 'не отклонена')
    sns.histplot(x,kde=True);plt.title(name);plt.xlabel('Сантиметры');plt.ylabel('Частота');show('iris_'+str(Gauss.counter));Gauss.counter+=1
    return p
Gauss.counter=0
iris_p={c:Gauss(iris[c],c) for c in iris.columns};display(pd.Series(iris_p,name='p χ²'))
worst=min(iris_p,key=iris_p.get);original=next(k for k,v in translations.items() if v==worst)
print('Минимальное p для:', worst)
species_p={species: Gauss(part[original], worst+' '+species) for species,part in iris_raw.groupby('Species_name')}
small_p=Gauss(iris_raw.loc[iris_raw.Species_name.eq('Setosa'),original],worst+' мелкие Setosa')
large_p=Gauss(iris_raw.loc[~iris_raw.Species_name.eq('Setosa'),original],worst+' крупные два сорта')

## Общий вывод
Смесь сортов может быть ненормальной даже при близких к нормальным распределениях отдельных сортов. Функция проверяет каждый сорт и объединение двух крупных сортов отдельно; результаты приведены непосредственно над выводом. Неотклонение нормальности не означает её доказанности. Во всех сравнениях нужно учитывать размер эффекта, выбор метода и множественность проверок, а не только порог 0,05.

In [ ]:
print('Выбранный признак:', worst)
for group, p in {**species_p, 'Мелкие (Setosa)': small_p, 'Крупные (два сорта)': large_p}.items():
    print(f"{group}: p={p:.6g}; H0 {'отклонена' if p < .05 else 'не отклонена'}.")
